# Turing machine enumeration

Here we give a way to count the Turing machines.
Under this function $\text{encode-TM}\colon \mathbb{N}\to T$, for every natural number there is an associated Turing machine
and for every Turing machine there is a natural number.
Because we will describe the function with Racket code, it is obviously computable.
That is, we will give an *acceptable numbering* of the Turing machines.

We start by defining some constants so that when they appear in the code, the intent is clear.

In [2]:
(define SEPARATOR "9")  ; inside of an instruction's representation
(define INTER-INSTRUCTION-SEPARATOR "99")  ; between instructions

(define LEFT #\L)
(define RIGHT #\R)

The basic idea is: a Turing machine is a set of instructions, which are 4-tuples.
To encode it, we fix some ordering and encode the instructions, separated by `INTER-INSTRUCTION-SEPARATOR`.
Each instruction encodes the four items separated by `SEPARATOR`.

To encode the instruction items, we must use numbers that cannot contain the digit $9$.  In the 4-tuple we must encode two state numbers, the present state and the next state, and two characters, the present input and the next action. (Because this is a simple demo, we won't worry about the possibility that the present input or next action are multi-character strings, and we also won;t worry unduly about distinguishing the symbols for `LEFT` and `RIGHT` from other characters.)

In [3]:
(define BASE-FOR-NUM-ENCODING 2) ; must be 2, or 3, ... or 8
(define BASE-FOR-CHAR-ENCODING 8) ; must be 2, or 3, ... or 8

When we are asked to decode an integer that doesn't make sense -- for instance if it contains an instruction encoding that does not include any `SEPARATOR`'s and so it doesn't break into four parts -- then we must return some default machine.

In [4]:
(define EMPTY-TURING_MACHINE '())

We will need the following utility function.
It takes a list and finds the `and` of the elements.

In [5]:
;; and-of-list  apply and to the list v
;;   (While (apply + '(1 2))) works, (apply and '(#t #f)) does not.  A bug, for sure.
(define (and-of-list v)
  (if (null? v)
      #t
      (and (car v)
           (and-of-list (cdr v)))))

## The parts of an instruction

Next we give routines to encode and decode the four items in an instruction.
The encoding routines take in either a natural number present state or next state, or a character for present input or next action, and output a string.
The decoding routines take in a string and output something of the relevant type.
(Strings are more convienent than natural numbers for the concatenation and element referencing that comes later.)

Note that both types of routine use a base other than decimal, because of the need to avoid the 
digit $9$, the `SEPARATOR`.
For the two number input cases, the routines use decimal.
For characters the routines use hexadecimal.
(The idea is that for most readers, hex is less familiar, but the character codes are also not familiar,
so it doesn't matter there and the space savings is nice.)

The `encode-xx` routines are more straightforward, because they are sure to be given known input.
However, the `decode-xx` routines must fail if the input is not sensible (for instance, if they must decode to 
a natural number but the string input just doesn't decode to that).
In each case, for such a parse failure the routine returns `#f`.

First is present state.  To encode or decode, we just need to convert between natural numbers and string representations. 
(Note that Racket's `string->number` routine returns `#f` if the string doesn't convert.)

In [6]:
;; encode-present-state  input positive integer, output encoding as string
(define (encode-present-state q)
  (number->string q BASE-FOR-NUM-ENCODING))

;; decode-present-state input a string, output natural number
;;  (if string is not suitable, output #f)
(define (decode-present-state s)
  (string->number s BASE-FOR-NUM-ENCODING))

The next state routines work the same way.

In [7]:
;; encode-next-state  input positive integer, output encoding as string
(define (encode-next-state q)
  (number->string q BASE-FOR-NUM-ENCODING))

;; decode-next-state input a string, output natural number
;;  (if string is not suitable, output #f)
(define (decode-next-state s)
  (string->number s BASE-FOR-NUM-ENCODING))

The tape symbols take a bit more care.
As to `encode-present-symbol`,
the `number->string` routine returns the *code point* of the character.
For instance, given the lower case a character, `#\a`, it returns $97$ (that's octal $141$), which is its slot in the Unicode tables.
(We will only ever use symbols that appear in the ASCII tables, which agree with Unicode.)

The `decode-present-symbol` routine is more involved.
The input string represents, in `BASE-FOR-CHAR-ENCODING`, the code point of some character.
It first verifies that the string isn't empty
(we don't elaborately vet the input, but this seemed to be a reasonable sanity check).
Then it interprets the string as number (which may fail),
and finally converts that number to a character.

In [9]:
;; encode-present-symbol  input character, output encoding as string
(define (encode-present-symbol s)
  (number->string (char->integer s) BASE-FOR-CHAR-ENCODING))

;; decode-present-symbol input a string, output character
;;  (if string is not suitable, output #f, but no check to avoid L or R)
(define (decode-present-symbol s)
  (if (= 0 (string-length s))
      #f
      (let ([char-code (string->number s BASE-FOR-CHAR-ENCODING)])
        (if (or (false? char-code)
                (>= 0 char-code))
            #f
            (integer->char char-code)))))

97

The `encode-next-action` and `decode-next-action` routines are similar.

In [ ]:
;; encode-next-action  input a character, output encoding as string representation of a number
;;   (note that the number avoids the digit 9, as it is the separator)
(define (encode-next-action s)
  (number->string (char->integer s) BASE-FOR-CHAR-ENCODING)) 

;; decode-next-action  input a string, output character
;;  (if string is not suitable, output #f)
(define (decode-next-action s)
  (if (= 0 (string-length s))
      #f
      (let ([char-code (string->number s BASE-FOR-CHAR-ENCODING)])
        (if (or (false? char-code)
                (>= 0 char-code))
            #f
            (integer->char char-code)))))


## Instructions

Next we encode and decode entire 4-tuple instructions.

The `encode-TM-instruction` routine takes in a list of 4 items, 
$(\text{present-state}\quad \text{present-character}\quad\text{next-action}\quad\text{next-state})$.
It uses the routines above to encode the items, and get a string that separates them with `SEPARATOR`.
The only twist is that this string represents a natural number so if the natural number should happen 
to start with $0$'s then there is the possibility of losing leading digits.
Thus the routine puts a `SEPARATOR` at the start, to cut that off.

As usual the `decode-TM-instruction` routine is a little bit more involved, because of the need to 
worry about input that doesn't parse (such as $`s`=0$.)
As usual, the routine follows the convention that if the input doesn't make a sensible instruction then it returns
`#f`.

In [ ]:
;; encode-TM-instruction  input list of four, output encoding as string
;;   Ensures no leading 0's, so full number is retained
(define (encode-TM-instruction inst)
  (let([present-state (first inst)]
       [present-symbol (second inst)]
       [next-action (third inst)]
       [next-state (fourth inst)])
    (string-append SEPARATOR (encode-present-state present-state)  ; leading sep so no leading 0
                   SEPARATOR (encode-present-symbol present-symbol)
                   SEPARATOR (encode-next-action next-action)
                   SEPARATOR (encode-next-state next-state))))

;; decode-TM-instruction  input string encoding instruction, return instruction (or empyty list if syntax doesn't match)
(define (decode-TM-instruction s)  
  (let([split-string (string-split s SEPARATOR)])  ; split-string consists of four strings
    (if (not (= 4 (length split-string)))
        #f
        (let ([this-state (decode-present-state (first split-string))]
              [this-input (decode-present-symbol (second split-string))]
              [next-action (decode-next-action (third split-string))]
              [next-state (decode-next-state (fourth split-string))])
          (if (not (and this-state this-input next-action next-state)) ; any fail to decode?
              #f
              (list this-state this-input next-action next-state))))))


## Whole machines

Most of the work is done.
The routine `encode-TM` just takes the strings generated from encoding the instructions, 
concatenates them, and returns the resulting integer. 
(Recall that earlier we ensured that this integer cannot have leading zeros.)

The routine `decode-TM` inputs an integer, breaks it into instructions, breaks each instruction into
four parts, and decodes all the parts.
If the integer doesn't divide up correctly (as with the input $`s`=0$) then this routine returns 
the default `EMPTY-TURING_MACHINE`.

(These two routines do not work with sets of instructions, for simplicity of presentation, but
what is here suffices to demonstrate the core idea.)

In [ ]:
;; encode-TM  input list of instructions, return integer encoding
(define (encode-TM tm)
  (if (null? tm)
      0 
      (string->number (string-join (map encode-TM-instruction tm) INTER-INSTRUCTION-SEPARATOR))))

;; decode-TM  input integer encoding of a Turing machine, return list representing that machine
(define (decode-TM s)
  (let ([encoded-instructions (string-split (number->string s) INTER-INSTRUCTION-SEPARATOR)])
    (let ([list-of-instructions (map decode-TM-instruction encoded-instructions)])
      (if (not (and-of-list list-of-instructions))
          EMPTY-TURING_MACHINE                                 ;
          (append list-of-instructions)))))

## Exercises

Use the cell below to do these exercises

1. Confirm that the symbol lower case a is encoded with the number $97$.

2. Sketch a numbering scheme using only blank and $1$.

2. What is the lowest-numbered non-trivial Turing machine?

3. Find the index of the successor function.

4. Recall that every computable function has infinitely many indices.  Give a different index for the successor function. 